In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

# classifiers
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier

# metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# ---------------------------------
# Load dataset
# ---------------------------------
df = pd.read_excel("Conf_Text_Labels.xlsx")

df = df.dropna(subset=['Text', 'Conf Label'])
df['Text'] = df['Text'].astype(str)
df['Conf Label'] = df['Conf Label'].astype(int)

X = df['Text']
y = df['Conf Label']


# ---------------------------------
# Train Test Split
# ---------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# ---------------------------------
# Classifiers
# ---------------------------------
models = {
    "SVM": LinearSVC(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Naive Bayes": MultinomialNB(),
    "AdaBoost": AdaBoostClassifier(),
    "MLP": MLPClassifier(max_iter=300)
}


# ---------------------------------
# Results storage
# ---------------------------------
results = []


# ---------------------------------
# Train & Evaluate each model
# ---------------------------------
for name, model in models.items():

    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('clf', model)
    ])

    # train
    pipeline.fit(X_train, y_train)

    # predictions
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    # metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    precision = precision_score(y_test, y_test_pred, average='weighted')
    recall = recall_score(y_test, y_test_pred, average='weighted')
    f1 = f1_score(y_test, y_test_pred, average='weighted')

    results.append([name, train_acc, test_acc, precision, recall, f1])


# ---------------------------------
# Create Results Table
# ---------------------------------
results_df = pd.DataFrame(results, columns=[
    "Model",
    "Train Accuracy",
    "Test Accuracy",
    "Precision",
    "Recall",
    "F1 Score"
])

print(results_df)

/opt/homebrew/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


           Model  Train Accuracy  Test Accuracy  Precision    Recall  F1 Score
0            SVM        0.894889       0.339374   0.314401  0.339374  0.322343
1  Decision Tree        0.980627       0.345964   0.327788  0.345964  0.329372
2  Random Forest        0.980627       0.387150   0.396104  0.387150  0.336398
3    Naive Bayes        0.572135       0.378913   0.305720  0.378913  0.291887
4       AdaBoost        0.394064       0.364086   0.314933  0.364086  0.317403
5            MLP        0.980214       0.342669   0.336831  0.342669  0.337950


/opt/homebrew/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

df = pd.read_excel("Conf_Text_Labels.xlsx")

# removing missing
df = df.dropna(subset=['Text', 'Conf Label'])

# converting text to string (IMPORTANT FIX)
df['Text'] = df['Text'].astype(str)

# removing blank text
df = df[df['Text'].str.strip() != ""]

# converting label to int
df['Conf Label'] = df['Conf Label'].astype(int)

X = df['Text']
y = df['Conf Label']

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

def tune_random_forest(X_train, y_train):

    # Pipeline: TF-IDF + RandomForest
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('rf', RandomForestClassifier(random_state=42))
    ])

    # Hyperparameter search space
    param_dist = {
        "tfidf__max_features": [1000, 3000],
        "tfidf__ngram_range": [(1,1), (1,2)],
        "rf__n_estimators": [50, 100, 150],
        "rf__max_depth": [None, 10, 20],
        "rf__min_samples_split": [2, 5, 10]
    }

    # RandomizedSearchCV
    random_search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_dist,
        n_iter=5,
        cv=3,
        random_state=42,
        n_jobs=1
    )

    random_search.fit(X_train, y_train)

    return random_search.best_estimator_, random_search.best_params_

best_model, best_params = tune_random_forest(X_train, y_train)

print("Best Parameters:", best_params)

Best Parameters: {'tfidf__ngram_range': (1, 2), 'tfidf__max_features': 3000, 'rf__n_estimators': 150, 'rf__min_samples_split': 2, 'rf__max_depth': 20}
